# mat-gram01

> Rebuild [IBM SMI-TED](https://huggingface.co/ibm-research/materials.smi-ted) as a [MAX](https://docs.modular.com/max/) custom architecture — one notebook at a time.


## What are we building?

A molecule arrives as a **SMILES** string — text like `CCO` for ethanol. We want a fixed **768-d embedding** we can serve from MAX.

That's the whole product. No language-model head. No property head inside the graph. Just:

```text
SMILES → tokens → MoLEncoder → AutoEncoder → 768-d vector
```

We'll write the port the way Jeremy Howard likes to teach: *explain a little, code a little, export a little*. Each notebook is a module. Read them in order; by the end you'll have a package MAX can load with `--custom-architectures`.


## Mental model

Two layers, same system:

```text
┌─────────────────────────────────────────────────────────────┐
│  MAX serve plumbing                                         │
│  arch → tokenizer → batch → pipeline model → weights        │
└────────────────────────────┬────────────────────────────────┘
                             │ calls
                             ▼
┌─────────────────────────────────────────────────────────────┐
│  Network math (graph)                                       │
│  tokens → MoLEncoder → AutoEncoder → 768-d embedding        │
└─────────────────────────────────────────────────────────────┘
```


## Notebook map

| Notebook | Module | What you'll learn |
|----------|--------|-------------------|
| [`00_config`](00_config.html) | `model_config` | Hyperparams from HuggingFace `config.json` |
| [`01_tokenizer`](01_tokenizer.html) | `tokenizer` | SMILES regex → token ids |
| [`02_weights`](02_weights.html) | `weight_adapters` | Which safetensor keys the encode graph keeps |
| [`03_graph`](03_graph.html) | `graph` | Linear attention, RoPE, encoder, AE, `build_graph` |
| [`04_batch`](04_batch.html) | `batch_processor` | Pad every sequence to `max_len=202` |
| [`05_model`](05_model.html) | `model` | Pipeline wrapper: load, compile, execute |
| [`06_arch`](06_arch.html) | `arch` | Register everything as `SupportedArchitecture` |
| [`07_package`](07_package.html) | `__init__` | Public exports (`ARCHITECTURES`, …) |
| [`08_finetune`](08_finetune.ipynb) | *(not exported)* | GPU IBM finetune → `finetune_ckpts/` → MAX export |


## How to use these notebooks

1. Work **top to bottom** inside a notebook. Don't skip the markdown — the code only makes sense with the story.
2. Cells marked `#| export` become the library under `mat_gram01/`.
3. After editing, run `nbdev_export` (or `nbdev_prepare`) so the package matches the notebooks.
4. The original package under `materials_smi_ted/` is the reference port; this literate rewrite lives in `nbs/` → `mat_gram01/`.


## Install (dev)

```sh
pip install -e .
nbdev_export
```


## How to use

Once exported, MAX finds the architecture the same way as the reference package:


In [ ]:
from mat_gram01 import ARCHITECTURES, smi_ted_arch
assert smi_ted_arch.name == "SmiTedModel"
ARCHITECTURES[0].name
